In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from astropy.utils.metadata.utils import dtype
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
from statsmodels.tools import categorical

# 2.load data

In [2]:
df = pd.read_csv("heart_disease_cleveland_edit.csv")

In [3]:
df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,Male,0,145.0,233.0,1,2,150,0,2.3,2,0,2,0
1,67,Male,3,160.0,286.0,0,2,108,1,1.5,1,3,1,1
2,67,Male,3,120.0,229.0,0,2,129,1,2.6,1,2,3,1
3,37,Male,2,130.0,250.0,0,0,187,0,3.5,2,0,1,0
4,41,Female,1,130.0,204.0,0,2,172,0,1.4,0,0,1,0


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       303 non-null    int64  
 1   sex       303 non-null    str    
 2   cp        303 non-null    int64  
 3   trestbps  298 non-null    float64
 4   chol      298 non-null    float64
 5   fbs       303 non-null    int64  
 6   restecg   303 non-null    int64  
 7   thalach   303 non-null    int64  
 8   exang     303 non-null    int64  
 9   oldpeak   303 non-null    float64
 10  slope     303 non-null    int64  
 11  ca        303 non-null    int64  
 12  thal      303 non-null    int64  
 13  target    303 non-null    int64  
dtypes: float64(3), int64(10), str(1)
memory usage: 34.6 KB


# 3.preprocessing data

In [5]:
# some datasets have unuseful columns like 'num'
df = df.drop(columns=['num'], errors='ignore')

In [6]:
X = df.drop('target', axis=1)
Y = df['target']

In [7]:
# checking null values
print('missing values:')
X.isna().sum()

missing values:


age         0
sex         0
cp          0
trestbps    5
chol        5
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          0
thal        0
dtype: int64

In [8]:
# replace '?' with NaN
X = X.replace('?', np.nan)

# 4. Convert values to numeric

In [10]:
# X = X.apply(pd.to_numeric, errors='ignore')

In [13]:
# select categorical data by data type

categorical_cols = [col for col in X.columns if X[col].dtype == 'str']
print(categorical_cols)


['sex']


In [14]:
# select numerical data by data type

numerical_cols = [col for col in X.columns if X[col].dtype in ['float64', 'int64']]
print(numerical_cols)

['age', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal']


In [15]:
numerical_transformer = Pipeline(steps=[
        'imputer', SimpleImputer(strategy='median')
])

In [16]:
categorical_transformer = Pipeline(steps=[
        'imputer', SimpleImputer(strategy='most_frequent'),
        'Encoder', OneHotEncoder(handle_unknown='ignore')
           ])

In [17]:
preprocesser = ColumnTransformer(transformers=[
    'num', numerical_transformer, numerical_cols,
    'cat', categorical_transformer, categorical_cols
])